# 11 - Results storyboard (read-only, committed files only)

This notebook regenerates the seven report figures from files **committed in git**:
- no Kaggle, no GPU, no raw CSVs and no Kaggle zips;
- nothing is trained and nothing is scored.

The logic lives in `make_figures.py`, which this notebook only calls.

**Every figure:**
- is labelled **VALIDATION**, **HELD-OUT TEST** or **DATA PROVENANCE**, with its seeds and source files;
- plots measured values only: mean marks with min-max bars over seeds, and no interpolation or smoothing;
- is saved to `docs/figures/` as a PNG, together with a `_data.csv` holding exactly the plotted numbers.

**Scope reminders:**
- The defended continual result is the **four-label** FU2 relabelling (figure 3), a post-hoc method.
- The 12-label continual goal is unsolved (figure 4).
- RF (figure 6) is a three-label, fixed-class limitation study in which repair never ran.

**One-command checks** (from the repository root):
- `python make_figures.py --execute-notebook` runs this notebook end to end;
- `python make_figures.py --check` confirms the plotted numbers match `docs/figures/*_data.csv` and the committed result
  tables.

In [ ]:
# 1. Setup: locate the repository root (this notebook lives in notebooks/) and import the figure code
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().resolve()
ROOT = ROOT if (ROOT / "make_figures.py").exists() else ROOT.parent
if not (ROOT / "make_figures.py").exists():
    raise SystemExit("Run this notebook from a clone of the repository (make_figures.py not found next to notebooks/).")
sys.path.insert(0, str(ROOT))
import make_figures as M
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 80)
print("repository:", ROOT)

In [ ]:
# 2. Artifact / provenance check: every committed input exists; sha256 (first 12 hex) shown for the record
try:
    prov = M.provenance()
except M.MissingArtifact as e:
    print("Cannot build the storyboard:\n", e)
    raise
print(f"{len(prov)} committed input files present")
display(prov)

## Data pipeline (provenance)

**Claim:**
- All 70,427,637 raw rows (18 files, 14 labels) were read once.
- Global deduplication and label-conflict removal shrink every class by orders of magnitude.
- BENIGN is 0.16% of the rows and is never capped.

**Limitation:**
- Training used capped subsets: 98,833 rows in the four-label setting and at most 159,595 (the 12-label oracle).
- There is no full-data training claim.

In [ ]:
data_fig1_data = M.fig1_data()  # writes docs/figures/fig1_data_pipeline.png and fig1_data_pipeline_data.csv
display(Image(filename=str(M.FIG / "fig1_data_pipeline.png")))
display(data_fig1_data.head(40))

## Pre-registered four-class sweep

**Claim** (pre-registered; seeds 42-44):
- The gate rejects more poisoned updates as f grows [validation].
- Soft repair lets the model learn LDAP and UDP for f <= 0.95 [test].

**Limitation:**
- At f = 1.0 soft repair fails (0/6).
- The gate alone keeps a stale model that never learns LDAP or UDP.
- Hollow dots show that gate-only results are bimodal across seeds.

In [ ]:
data_fig2_sweep = M.fig2_sweep()  # writes docs/figures/fig2_sweep.png and fig2_sweep_data.csv
display(Image(filename=str(M.FIG / "fig2_sweep.png")))
display(data_fig2_sweep.head(40))

## FU2: four-class defence at 100% flips (PRIMARY, post hoc)

**Claim** [test]: at f = 1.0, teacher relabelling learns all four classes in 3/3 seeds, within about 0.01 of clean, with a benign FAR of about 0.0011.

**Limitation:**
- The method is post hoc: chosen after the sweep's diagnostic, and E1 replays that diagnostic.
- Four labels, one attacked class, 3 seeds.
- A FAR of 0 for gate-only is not safety: the stale model calls LDAP and UDP benign.

In [ ]:
data_fig3_fu2 = M.fig3_fu2()  # writes docs/figures/fig3_fu2_defence.png and fig3_fu2_defence_data.csv
display(Image(filename=str(M.FIG / "fig3_fu2_defence.png")))
display(data_fig3_fu2.head(40))

## 12-label continual vs joint oracle (LIMITATION)

**Claim** [test]:
- The CII continual 12-label models retain 6 of 12 classes.
- A joint oracle reaches all 12, but 8 or fewer reach recall 0.80.

**Limitation:**
- The oracle is one jointly trained diagnostic model, not a continual or defended result.
- The 12-label continual goal is unsolved.
- Continual runs are single-seed.

In [ ]:
data_fig4_oracle = M.fig4_oracle()  # writes docs/figures/fig4_12label_vs_oracle.png and fig4_12label_vs_oracle_data.csv
display(Image(filename=str(M.FIG / "fig4_12label_vs_oracle.png")))
display(data_fig4_oracle.head(40))

## RC / RCS clean pilots (validation only, stopped)

**Claim** [validation]: every clean rejection is an old non-Syn class forgotten when an overlapping class arrives.

**Limitation:**
- The registered stopping rules ended both experiments before any poisoned run.
- There is no test result to plot.
- Only three schedules were tried.

In [ ]:
data_fig5_pilots = M.fig5_pilots()  # writes docs/figures/fig5_rc_rcs_pilots.png and fig5_rc_rcs_pilots_data.csv
display(Image(filename=str(M.FIG / "fig5_rc_rcs_pilots.png")))
display(data_fig5_pilots.head(40))

## RF: confidence loss without recall loss (LIMITATION, post hoc)

**Claim** (validation and test in separate panels):
- At 10% more flips per update (60% after six updates), Syn recall holds at about 0.994.
- Mean Syn confidence falls to about 0.77.
- No update was rejected, so repair never triggered.

**Limitation:**
- The confidence loss was not repaired.
- Three fixed classes; traffic refresh, not new-class arrival.

In [ ]:
data_fig6_rf = M.fig6_rf()  # writes docs/figures/fig6_rf_trajectory.png and fig6_rf_trajectory_data.csv
display(Image(filename=str(M.FIG / "fig6_rf_trajectory.png")))
display(data_fig6_rf.head(40))

## Experiment status

Which experiments were pre-registered, post hoc, validated, tested once or stopped. Each cell is asserted against the evidence folders present in git.

In [ ]:
data_fig7_status = M.fig7_status()  # writes docs/figures/fig7_status.png and fig7_status_data.csv
display(Image(filename=str(M.FIG / "fig7_status.png")))
display(data_fig7_status.head(40))

## Plotted numbers vs committed result tables

In [ ]:
# 3. Every plotted number is checked against an independently read committed table (assertions raise on mismatch)
data = {f.__name__: globals()[f"data_{f.__name__}"] for f in M.FIGURES}
for m in M.verify(data):
    print("OK", m)
print("\nfigure files:", sorted(p.name for p in M.FIG.glob("fig*.png")))